# ST 554 Project 2
---
Authored by: Jamie Loring

## Part 1 (continued)
We now use the class created in the `Loring_Project2.py` file on some data!

## Part 2
This part focuses on using `spark` to analyze NFL data.

The code below imports all required modules for this entire section and runs objects needed to suppress/manage unnecessary warnings.

In [ ]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

import pandas as pd
import numpy as np
import pyspark.pandas as ps

ps.set_option('compute.fail_on_ansi_mode', False)

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

#code needed to manage package warnings -- this spark object is used for both pandas-on-spark and Spark SQL
spark = SparkSession.builder.appName('my_app').config('spark.sql.ansi.enabled', 'false').getOrCreate()

#### `pandas`-on-Spark

The code below reads in the weekly NFL data from the main filepath in JupyterHub.

In [ ]:
NFL = ps.read_csv("weekly_nfl_data.csv")

Next, we use `.head()` to check out the first 5 rows of the `NFL` dataframe.

In [ ]:
NFL.head()

The code below reports all the column names of the `NFL` dataframe.

In [ ]:
NFL.columns

Next, we only want to look at QB stats for the seasons 2005 to 2023 (inclusive). Additionally:
- We only want the following columns -
    - `player_display_name`
    - `season`
    - `week`
    - `completions`
    - `attempts`
    - `passing_yards`
    - `passing_tds`
    - `interceptions`
- For each combination of `player_display_name` and `season`, we will find the *sum* and *mean* of each of the statistical quantities in this subsetted dataset
- We will create two new variables by player/season combination:
    - `completion_percentage` = (sum of completions) / (sum of attempts)
    - `td_int_ratio` = (sum of passing tds) / (sum of interceptions)

The code below subsets `NFL` to look at regular season QB stats, specifically for seasons 2005 to 2023, inclusive. The code also subsets the columns of interest. This new object is called `NFL_sub1`.

In [ ]:
NFL_sub1 = NFL.loc[(NFL['position'] == 'QB') & 
                   (NFL['season_type'] == 'REG') & 
                   (NFL['season'].between(2005, 2023))] \
              [["player_display_name", "season", "week", "completions",
                "attempts", "passing_yards", "passing_tds", "interceptions"]]

Next, the code below calculates the sum and mean for each of the statistical quantities in this subsetted data for each combination of `player_display_name` and `season`. This new object is called `NFL_sub2`.

In [ ]:
NFL_sub2 = NFL_sub1.groupby(["player_display_name", "season"]) \
                    [["completions", "attempts", "passing_yards",
                      "passing_tds", "interceptions"]].agg(['mean', 'sum'])

Finally, we create the two new variables that were previously described! Then, they are added to our already subsetted dataframe (`NFL_sub2`).

In [ ]:
cp = NFL_sub2["completions", "sum"] / NFL_sub2["attempts", "sum"] #creates completion_percentage variable
NFL_sub2["completion_percentage"] = cp #adds completion_percentage variable to NFL_sub2

tdir = NFL_sub2["passing_tds", "sum"] / NFL_sub2["interceptions", "sum"] #creates td_int_ratio variable
NFL_sub2["td_int_ratio"] = tdir #adds td_int_ratio variable to NFL_sub2

NFL_sub2.head() #previews final df

The final results of the above are saved in the `NFL_sub2` object.

Now, we take `NFL_sub2` and do the following:
- Subset the rows to only include player/season combinations where the sum of attempts is at least 50
- Sort the rows descending by `completion_percentage` and report the first 40 values
- Sort the rows descending by `td_int_ratio` and report the first 40 values

First, we subset the `NFL_sub2` object to only include the rows where the player/season combination sum of attempts is at least 50. This new object is called `NFL_sub3`.

In [ ]:
NFL_sub3 = NFL_sub2.loc[NFL_sub2["attempts", "sum"] >= 50]

Next, we sort the rows of `NFL_sub3` descending by `completion_percentage` and report the first 40 values.

In [ ]:
NFL_sub3.sort_values("completion_percentage", ascending=False).head(40)

Finally, independent of the previous step, we again sort the rows of `NFL_sub3`, this time descending by `td_int_ratio` and report the first 40 values.

In [ ]:
NFL_sub3.sort_values("td_int_ratio", ascending=False).head(40)

#### Spark SQL

The code below reads in the weekly NFL data from the main filepath in JupyterHub.

In [ ]:
NFL_spark = spark.read.load("weekly_nfl_data.csv",
                            format="csv",
                            sep=",",
                            inferSchema="true",
                            header="true")

Next, we convert to a `pandas`-on-Spark dataframe and use `head()` to check out the first 5 rows of the `NFL_spark` dataframe.

**Note:** This conversion method will be used on all outputs of the dataframes in this section so our results look a bit more presentable!

In [ ]:
NFL_spark.pandas_api().head()

The code below reports all the column names of the `NFL_spark` dataframe.

In [ ]:
NFL_spark.columns

Next, we only want to look at QB stats for the seasons 2005 to 2023 (inclusive). Additionally:
- We only want the following columns -
    - `player_display_name`
    - `season`
    - `week`
    - `completions`
    - `attempts`
    - `passing_yards`
    - `passing_tds`
    - `interceptions`
- For each combination of `player_display_name` and `season`, we will find the *sum* and *mean* of each of the statistical quantities in this subsetted dataset
- We will create two new variables by player/season combination:
    - `completion_percentage` = (sum of completions) / (sum of attempts)
    - `td_int_ratio` = (sum of passing tds) / (sum of interceptions)

The code below subsets `NFL_spark` to meet all of the above criteria and also adds the two new variables. This is all accomplished in the one code chunk below. The results are saved as an object called `NFL_spark_ss`.

In [ ]:
NFL_spark_ss = NFL_spark.filter((NFL_spark.position == 'QB') & (NFL_spark.season_type == 'REG') &
                                (NFL_spark.season.between(2005,2023))) \
                        .select(["player_display_name", "season", "week", "completions",
                                 "attempts", "passing_yards", "passing_tds", "interceptions"]) \
                        .groupby(["player_display_name", "season"]) \
                        .agg(sum("completions"), avg("completions"), sum("attempts"), avg("attempts"),
                             sum("passing_yards"), avg("passing_yards"), sum("passing_tds"), avg("passing_tds"),
                             sum("interceptions"), avg("interceptions")) \
                        .withColumn("completion_percentage", col("sum(completions)") / col("sum(attempts)")) \
                        .withColumn("td_int_ratio", col("sum(passing_tds)") / col("sum(interceptions)"))

NFL_spark_ss.pandas_api().head() #previews resulting df

Now, we take `NFL_spark_ss` and do the following:
- Subset the rows to only include player/season combinations where the sum of attempts is at least 50.
- Sort the rows descending by completion_percentage and report the first 40 values!
- Sort the rows descending by td_int_ratio and report the first 40 values!

First, we show `NFL_spark_ss` subsetted to only include player/season combinations where the sum of attempts is at least 50. Then, we sort the rows descending by `completion_percentage` and report the first 40 values. This is all done in one code chunk.

In [ ]:
NFL_spark_ss.filter(col("sum(attempts)") >= 50) \
            .sort(NFL_spark_ss.completion_percentage, ascending = False) \
            .pandas_api() \
            .head(40)

Finally, we show `NFL_spark_ss` subsetted to only include player/season combinations where the sum of attempts is at least 50. Then, we sort the rows descending by `td_int_ratio` and report the first 40 values. This is all done in one code chunk.

In [ ]:
NFL_spark_ss.filter(col("sum(attempts)") >= 50) \
            .sort(NFL_spark_ss.td_int_ratio, ascending = False) \
            .pandas_api() \
            .head(40)

**Important Note:** The `td_int_ratio` values are treated differently between `pandas`-on-Spark and Spark SQL. With `pandas`-on-Spark, `NaN` is the result if zero is being divided by zero when creating the ratio. If a positive number is divided by 0, then the result is `inf`. With Spark SQL, however, regardless of what the numerator is in the division fraction, `NaN` is the result if there is division by 0. This is why the (descending) sorted dataframes between `pandas`-on-Spark and Spark SQL look a bit different. The `inf` results are included in the `pandas`-on-Spark output, but those equivalent rows with Spark SQL are not because they are stored as `NaN`.